# Sensitivity Analysis

Reads the sweeps written by `../custom_drivers/sensitivity_simulation.ipynb`. Every interval is a
**scene** bootstrap. Scenes are themselves uniform draws over P.1410 (α, β, γ), so 38.811
scoring bins scenes into the nearest P.1410 class and compares each bin with its own table.
Continuous factors are judged by their **swing**: the change in a scene-level output across
the factor's full uniform range, from the paired per-scene response Δ(x). An effect counts
only if its CI excludes 0 and it exceeds `K_FLOOR` × the Phase 0 solver-noise floor.
Methods: docs/sensitivity.md, docs/analysis.md.

In [ ]:
import os, sys, re, json, glob, pickle, warnings
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(REPO.parent))
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from Ray_Tracing.src.utils import sensitivity as S

SWEEP_ROOT = REPO / "Results" / "sensitivity"
OUT_DIR    = SWEEP_ROOT / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
B          = 500              # bootstrap replicates
K_FLOOR    = 2.0              # effect must exceed K_FLOOR x replicate std
OUTPUTS    = ("K_dB", "lgDS", "is_los", "K_fit_dB")
KEY_STATS  = ["mu_K_dB", "sigma_K_dB", "K_rom_dB", "mu_lgDS_LOS", "sigma_lgDS_LOS",
              "mu_lgDS_NLOS", "P_LoS", "mu_lgASA_LOS", "mu_lgZSA_LOS"]


def phase(name, **kw):
    d = SWEEP_ROOT / name
    return S.load_sweep(str(d), **kw) if d.is_dir() else None


def variant_specs(name):
    out = {}
    for vj in glob.glob(str(SWEEP_ROOT / name / "*" / "variant.json")):
        v = json.load(open(vj))
        out[re.sub(r"^[\d.]+GHz__", "", v["name"])] = v
    return out


def ghz(f):
    return f"{f / 1e9:g} GHz"


def err(est, lo, hi):
    # Percentile CIs need not bracket the point estimate; clip for errorbar().
    return [np.clip(est - lo, 0, None), np.clip(hi - est, 0, None)]

## Phase 0 — baseline, noise floor, estimators, distance to 38.811

In [ ]:
df0 = phase("phase0")
REPLICATES = sorted(df0.variant.unique())          # baseline + rep1..n
base0 = df0[df0.variant == "baseline"].copy()
base0["p1410"] = S.p1410_class(base0)

summary0    = S.summary_table(base0, B=B)
floor       = S.noise_floor(df0, REPLICATES)               # ensemble statistics
floor_scene = S.noise_floor_scene(df0, REPLICATES, OUTPUTS)  # scene-level means (for swings)
summary0.to_csv(OUT_DIR / "phase0_baseline_summary.csv", index=False)
floor.to_csv(OUT_DIR / "phase0_noise_floor.csv")
floor_scene.to_csv(OUT_DIR / "phase0_noise_floor_scene.csv")

fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, st in zip(axs, ["mu_K_dB", "mu_lgDS_LOS", "P_LoS"]):
    for f, g in summary0[summary0.stat == st].groupby("frequency"):
        g = g.sort_values("elevation")
        ax.errorbar(g.elevation, g.est, yerr=err(g.est, g.lo, g.hi), capsize=3, marker="o", label=ghz(f))
    ax.set(title=st, xlabel="elevation [°]")
axs[0].legend()
plt.tight_layout(); plt.show()
print("scenes per P.1410 class:", base0.groupby("p1410").scene_seed.nunique().to_dict())
floor[[c for c in KEY_STATS if c in floor]].round(3)

In [ ]:
# Realised scene parameters vs their uniform draws, and the orbit geometry actually sampled.
sc = base0.groupby("scene_seed").first()
fig, axs = plt.subplots(1, 4, figsize=(15, 3))
for ax, (nom, real) in zip(axs[:3], (("geo.alpha", "itu_alpha"), ("geo.beta", "itu_beta"),
                                      ("geo.gamma", "itu_gamma_fit"))):
    ax.scatter(sc[nom], sc[real], s=8); lim = [sc[nom].min(), sc[nom].max()]
    ax.plot(lim, lim, "k--", lw=.8); ax.set(xlabel=f"drawn {nom}", ylabel=f"realised {real}")
axs[3].hist(np.degrees(np.arcsin(np.sin(np.radians(sc.gs_lat)))), bins=15)
axs[3].set(xlabel="ground-station latitude [°]", title="area-uniform ground points")
plt.tight_layout(); plt.show()

In [ ]:
# Sweep #4: is the K trend vs 38.811 an estimator artefact?
est0 = S.estimator_table(base0, by=("p1410", "frequency", "elevation"))
est0.to_csv(OUT_DIR / "phase0_k_estimators.csv", index=False)
est0[est0.state == "LOS"].round(2)

In [ ]:
# Distribution distance, each P.1410 class against its own 38.811 table.
dist0 = S.dist_metrics(base0, scenario="by_class")
dist0.to_csv(OUT_DIR / "phase0_dist_38811.csv", index=False)
dist0.pivot_table(index=["p1410", "frequency", "elevation"], columns=["state", "lsp"], values="w1").round(2)

## Phase 1 — interaction depth and mechanisms

In [ ]:
df1 = phase("phase1")
if df1 is not None:
    s1 = S.summary_table(df1, B=B)
    s1.to_csv(OUT_DIR / "phase1_summary.csv", index=False)
    parts = s1.variant.str.extract(r"depth(\d)_diffuse(\d)_diffr(\d)").astype(int)
    s1 = s1.assign(depth=parts[0], diffuse=parts[1], diffr=parts[2])
    for st in ["mu_K_dB", "mu_lgDS_LOS", "mu_lgDS_NLOS", "mu_n_paths"]:
        g = s1[s1.stat == st]
        fig, axs = plt.subplots(1, g.frequency.nunique() * g.elevation.nunique(),
                                figsize=(14, 3), sharey=True, squeeze=False)
        for ax, ((f, e), sub) in zip(axs[0], g.groupby(["frequency", "elevation"])):
            for (dfu, dfr), c in sub.groupby(["diffuse", "diffr"]):
                c = c.sort_values("depth")
                ax.errorbar(c.depth, c.est, yerr=err(c.est, c.lo, c.hi), marker="o", capsize=2,
                            label=f"diffuse={dfu} diffr={dfr}")
            ax.set(title=f"{ghz(f)} {e:g}°", xlabel="max depth")
        axs[0, 0].set_ylabel(st); axs[0, -1].legend(fontsize=6)
        plt.tight_layout(); plt.show()
    full = s1[(s1.diffuse == 1) & (s1.diffr == 1) & s1.stat.str.startswith("share_P_d")]
    display(full.pivot_table(index=["frequency", "elevation", "depth"], columns="stat",
                             values="est").round(3))

## Phase 2 — uniform one-at-a-time responses

For each `oat_*` variant, every scene drew the factor from its uniform range. The paired
per-scene Δ(x) (variant − baseline, same scene/orbit/UEs/rays) is fitted with a quadratic
through Δ(x0) = 0; **swing** = max − min of that fit over the range. Categorical variants
(`cat_*`) are paired Δ of the ensemble statistics.

In [ ]:
def responses(name):
    d = phase(name)
    if d is None:
        return None, None
    specs = variant_specs(name)
    d = pd.concat([d, df0[df0.variant == "baseline"]], ignore_index=True)
    rows = []
    for vname, v in specs.items():
        if "oat" in v:
            o = v["oat"]
            r = S.oat_response(d[d.variant.isin([vname, "baseline"])], "baseline", o["factor"],
                               tuple(o["spec"]), o["x0"], OUTPUTS, floor=floor_scene, B=B,
                               k_floor=K_FLOOR)
            rows.append(r.assign(variant=vname))
    resp = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    cats = [n for n in specs if "oat" not in specs[n]]
    cat = (S.oat_effects(d[d.variant.isin(cats + ["baseline"])], "baseline", floor=floor, B=B,
                         k_floor=K_FLOOR) if cats else pd.DataFrame())
    resp.to_csv(OUT_DIR / f"{name}_swings.csv", index=False)
    cat.to_csv(OUT_DIR / f"{name}_categorical.csv", index=False)
    return resp, cat


def swing_tornado(resp, output, elevation, title):
    g = resp[(resp.output == output) & (resp.elevation == elevation)]
    fs = sorted(g.frequency.unique())
    fig, axs = plt.subplots(1, len(fs), figsize=(6 * len(fs), 0.28 * g.factor.nunique() + 1.5),
                            sharey=True, squeeze=False)
    order = g.groupby("factor").swing.max().sort_values().index
    for ax, f in zip(axs[0], fs):
        h = g[g.frequency == f].set_index("factor").reindex(order)
        y = np.arange(len(h))
        ax.barh(y, h.swing, color=np.where(h.significant.fillna(False).astype(bool), "C3", "0.6"))
        ax.errorbar(h.swing, y, xerr=err(h.swing, h.swing_lo, h.swing_hi), fmt="none", ecolor="k", lw=.8)
        ax.axvspan(0, K_FLOOR * h.noise.max(), color="0.9", zorder=0)
        ax.set(yticks=y, yticklabels=h.index, title=ghz(f), xlabel=f"swing of {output} over range")
    fig.suptitle(f"{title}, elevation {elevation:g}° (red: significant; grey band: noise floor)")
    plt.tight_layout(); plt.show()


resp_mat, cat_mat   = responses("phase2_materials")
resp_link, cat_link = responses("phase2_link")
for resp, t in ((resp_mat, "Materials"), (resp_link, "Link")):
    if resp is not None and len(resp):
        for e in sorted(resp.elevation.unique()):
            swing_tornado(resp, "K_dB", e, t)

In [ ]:
# Response curves for the significant factors: per-scene paired Δ vs drawn value.
for name, resp in (("phase2_materials", resp_mat), ("phase2_link", resp_link)):
    if resp is None or not len(resp):
        continue
    d = pd.concat([phase(name), df0[df0.variant == "baseline"]], ignore_index=True)
    sig = resp[resp.significant & (resp.output == "K_dB")].drop_duplicates("factor").factor
    for fac in sig:
        v = d[d[fac].notna()]
        e0 = sorted(v.elevation.unique())[0]
        tv = S.scene_table(v[v.elevation == e0], [fac], ("K_dB",)).set_index("scene_seed")
        tb = S.scene_table(d[(d.variant == "baseline") & (d.elevation == e0)], [], ("K_dB",)).set_index("scene_seed")
        c = tv.index.intersection(tb.index)
        plt.figure(figsize=(4, 3)); plt.scatter(tv.loc[c, fac], tv.loc[c, "K_dB"] - tb.loc[c, "K_dB"], s=8)
        plt.axhline(0, color="k", lw=.6); plt.xlabel(fac); plt.ylabel("Δ scene-mean K [dB]")
        plt.title(f"{e0:g}° (first frequency group)"); plt.tight_layout(); plt.show()

In [ ]:
for cat, t in ((cat_mat, "materials"), (cat_link, "link")):
    if cat is not None and len(cat):
        sig = cat[cat.significant & cat.stat.isin(KEY_STATS)]
        display(sig.pivot_table(index="variant", columns=["frequency", "elevation", "stat"],
                                values="delta").round(2).dropna(axis=1, how="all"))

## Phase 3 — where the variance comes from

Within-scene (UE position), between-scene explained by the scene draws (realised α, β, γ
and satellite azimuth relative to the street grid), and residual between-scene randomness.

In [ ]:
df3 = phase("phase3")
if df3 is not None:
    df3["rel_az_deg"] = (df3.sat_az_deg - df3.heading_deg) % 90.0
    facs = ["itu_alpha", "itu_beta", "itu_gamma_fit", "rel_az_deg"]
    rows = []
    for (f, e), g in df3.groupby(["frequency", "elevation"]):
        rows.append(S.geometry_variance(g, facs).assign(frequency=f, elevation=e))
    gv = pd.concat(rows, ignore_index=True)
    gv.to_csv(OUT_DIR / "phase3_variance_split.csv", index=False)
    if gv.empty:
        print("phase3: fewer than 10 scenes with finite values per group; no split")
    for lsp, g in gv.groupby("lsp"):
        g = g.sort_values(["frequency", "elevation"])
        lab = [f"{ghz(f)}\n{e:g}°" for f, e in zip(g.frequency, g.elevation)]
        fig, ax = plt.subplots(figsize=(7, 2.8))
        bot = np.zeros(len(g))
        for col, c in (("share_within", "C0"), ("share_between_explained", "C2"),
                       ("share_between_residual", "C1")):
            ax.bar(lab, g[col], bottom=bot, color=c, label=col.replace("share_", "")); bot += g[col].to_numpy()
        ax.set(title=f"{lsp} variance split", ylim=(0, 1)); ax.legend(fontsize=7, loc="upper right")
        plt.tight_layout(); plt.show()

## Phase 4 — global (Sobol) indices

Legendre PCE on scene-level means over the full uniform design (scene draws + materials +
link), one study per carrier and elevation band. `r2_cv` is the share of scene-to-scene
variance the factors explain. Indices are conditional on the uniform ranges.

In [ ]:
df4 = phase("phase4")
BANDS = {"low (10-20°)": (10, 20), "high (60-80°)": (60, 80)}
if df4 is not None:
    with open(next((SWEEP_ROOT / "phase4").glob("*lhs/scenes_manifest.pkl")), "rb") as fh:
        design_spec = {k: tuple(v) for k, v in pickle.load(fh)["design"]["spec"].items()}
    sob = []
    for f, gf in df4.groupby("frequency"):
        for band, (lo, hi) in BANDS.items():
            g = gf[(gf.elevation >= lo) & (gf.elevation <= hi)]
            if g.empty:
                continue
            t = S.sobol_from_design(g, design_spec, outputs=("K_dB", "lgDS", "is_los"), degree=2, B=200)
            sob.append(t.assign(frequency=f, band=band))
    sob = pd.concat(sob, ignore_index=True)
    sob.to_csv(OUT_DIR / "phase4_sobol.csv", index=False)
    for (f, band, out), g in sob.groupby(["frequency", "band", "output"]):
        g = g.sort_values("ST")
        fig, ax = plt.subplots(figsize=(6, 0.3 * len(g) + 1))
        ax.barh(g.factor, g.ST, xerr=err(g.ST, g.ST_lo, g.ST_hi), color="C0", label="total")
        ax.barh(g.factor, g.S1, color="C1", alpha=.7, label="first order")
        ax.set(title=f"{ghz(f)} {band} {out}  (R²cv={g.r2_cv.iloc[0]:.2f})", xlim=(0, 1))
        ax.legend(fontsize=7); plt.tight_layout(); plt.show()